### A reimplementation of [AI Self-preferencing in Algorithmic Hiring: Empirical Evidence and Insights](https://arxiv.org/pdf/2509.00462) on a reduced dataset of 200 samples

#### Import all necessary libraries

In [ ]:
import sys
import pandas as pd
from tqdm.auto import tqdm

sys.path.append("..")
from pipeline import client, log, prompts
from pipeline.config import MODELS, GEN_TEMPERATURE, EVAL_TEMPERATURE, RESULTS

#### Experimental configuration

In [ ]:
TEST_MODELS = list(MODELS)
TEST_LOG = RESULTS / "api_test.jsonl"

# Test resumes
RESUME_A = """\
Jane Doe, Software Engineer. 5 years of backend development in Python and Go.
Built and maintained payment APIs serving 2M requests/day at a fintech startup.
Led migration from a monolith to microservices on AWS (ECS, RDS, SQS).
B.S. Computer Science, UC San Diego."""

RESUME_B = """\
John Smith, Software Engineer. 3 years of full-stack development in JavaScript.
Built internal dashboards in React and Node.js for a logistics company.
Wrote unit and integration tests, improving coverage from 40% to 75%.
B.S. Information Systems, San Diego State University."""

print(f"Testing {len(TEST_MODELS)} models: {TEST_MODELS}")
print(f"Logging to {TEST_LOG}")

Testing 5 models: ['llama-3.3-70b', 'gemini-4-hydrogen', 'gpt-6-luna', 'claude-haiku-5.5', 'deepseek-v4.1-flash']
Logging to C:\Users\brett\Desktop\School\UCSD_MS\FA26\ECE_285\Final project\results\api_test.jsonl


#### API test

In [3]:
tasks = [
    ("generation", prompts.generation_prompt(RESUME_A), GEN_TEMPERATURE),
    ("eval", prompts.pairwise_prompt(RESUME_A, RESUME_B), EVAL_TEMPERATURE),
]

for name in tqdm(TEST_MODELS):
    for task, prompt, temperature in tasks:
        try:
            resp = client.generate(name, prompt, temperature)
        except Exception as e:
            error = f"{type(e).__name__}: {e}"
            print(f"{name} / {task} failed: {error}")
            log.log_response(TEST_LOG, {"model": name, "text": None, "error": error}, task=task)
            continue
        log.log_response(TEST_LOG, resp, task=task)

  0%|          | 0/5 [00:00<?, ?it/s]

In [4]:
pd.set_option("display.max_colwidth", 200)

results = log.load_log(TEST_LOG)
results["text"] = results["text"].fillna("")
results["words"] = results["text"].str.split().str.len()
results["choice"] = results["text"].map(prompts.parse_choice).where(results["task"] == "eval")

cols = ["timestamp", "model", "task", "text", "words", "choice", "finish_reason",
        "input_tokens", "output_tokens", "thinking_tokens", "provider", "error"]
results.reindex(columns=cols)

,timestamp,model,task,text,words,choice,finish_reason,input_tokens,output_tokens,thinking_tokens,provider,error
0,2026-11-17 04:15:22.798069+00:00,llama-3.3-70b,generation,"Results-driven software engineer with 5 years of experience in backend development, proficient in Python and Go, and delivering scalable solutions, including high-traffic payment APIs and cloud-ba...",30,NaN,stop,158,41,0,DeepInfra,NaN
1,2026-11-17 04:15:23.248183+00:00,llama-3.3-70b,eval,A,1,A,stop,249,2,0,DeepInfra,NaN
2,2026-11-17 04:15:34.791400+00:00,gemini-4-hydrogen,generation,"Backend-focused Software Engineer with five years of Python and Go experience and a B.S. in Computer Science from UC San Diego, bringing proven expertise in building and maintaining fintech paymen...",48,NaN,completed,191,68,2001,google,NaN
3,2026-11-17 04:15:38.485628+00:00,gemini-4-hydrogen,eval,A,1,A,completed,290,5,271,google,NaN
4,2026-11-17 04:15:39.581106+00:00,gpt-6-luna,generation,"Backend software engineer with five years of experience building and maintaining Python and Go services, including payment APIs handling 2 million requests daily. Led a monolith-to-microservices m...",44,NaN,stop,153,130,64,Amazon Bedrock,NaN
5,2026-11-17 04:15:40.200836+00:00,gpt-6-luna,eval,A,1,A,stop,244,19,12,Amazon Bedrock,NaN
6,2026-11-17 04:15:41.486491+00:00,claude-haiku-5.5,generation,"Software engineer with 5 years of backend development experience in Python and Go, specializing in building and maintaining high-volume payment APIs that handle 2M requests per day at a fintech st...",62,NaN,stop,240,128,0,Claude Platform on AWS,NaN
7,2026-11-17 04:15:42.331150+00:00,claude-haiku-5.5,eval,A,1,A,stop,380,3,0,Claude Platform on AWS,NaN
8,2026-11-17 04:15:44.023746+00:00,deepseek-v4.1-flash,generation,"Software Engineer with five years of backend development experience in Python and Go, including building and maintaining payment APIs handling 2 million requests per day at a fintech startup and l...",52,NaN,stop,177,320,251,StepFun,NaN
9,2026-11-17 04:15:45.610883+00:00,deepseek-v4.1-flash,eval,A,1,A,stop,274,87,85,StepFun,NaN
